# Experiment 018: seed-1 recovery test under a revived schedule (Colab)

Continues the 006 seed 1 run from its saved step-500 state to step 1000. The
resumed scheduler applies one zero-rate boundary step at 501, then the
rebuilt 1000-step schedule revives the rate to about half peak from step 502,
so this is a recovery phase, not plain extra training, full framing in the plan.

Pre-registered rules: equivalence = paired 95% ci entirely inside +-2pt;
material change = |delta| >= 2pt and mcnemar p < .05; otherwise inconclusive.
Reward observable = mean(steps 901-1000) minus mean(501-600), labels at
+-0.03. Probe uses the same equivalence rule.

## 0. Compute units and mount Drive

Fill `cu_before` and `unit_rate` now.

In [1]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 23.04   # fill the current balance, this also closes the previous open session
unit_rate = 5.3    # units per hour from the panel
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '018_ext_seed1', 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260727_162656 | cu_before = 23.04
Mounted at /content/drive
closed previous session: 20260727_091529
saved open session: 20260727_162656
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: any A100, 40GB preferred

In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
assert 'A100' in gpu, 'pick an a100 runtime'
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram {total_gb:.1f} GiB' + ('' if total_gb < 60 else ' (80gb variant, acceptable, gets recorded)'))

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-8d18da26-812d-781b-6c0d-7d16847148cc)
total vram 39.5 GiB


## 2. Clone or update the repo (automatic)

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # instead of using the stale clone
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 2148, done.
remote: Counting objects: 100% (627/627), done.
remote: Compressing objects: 100% (390/390), done.
remote: Total 2148 (delta 390), reused 468 (delta 237), pack-reused 1521 (from 1)
Receiving objects: 100% (2148/2148), 67.52 MiB | 19.48 MiB/s, done.
Resolving deltas: 100% (1297/1297), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 156.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 47.6 MB/s eta 0:00:00
deps installed


## 4. W&B key (automatic)

Reminder: colab secrets are granted per notebook, switch on access for
`WANDB_API_KEY` under the key icon before running. This key is only used for experiment tracking.


In [5]:
import getpass
import os

wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None

if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')

os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check the Drive record and free space

In [6]:
import os, shutil
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 10, 'need some drive headroom for the continuation artifacts'

open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260727_162656.json
drive free: 179 GB


## 6. Seed the continuation directory (hash-verified copies)

Copies the exact 006 seed-1 merged base and checkpoint-500 into the fresh 018
directory, asserting every file hash matches its source. Skipped automatically
once 018 has progressed past step 500 (resume), and it never overwrites newer
progress with the step-500 seed.

In [7]:
import hashlib, json, os, shutil, glob, time

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

D = '/content/drive/MyDrive/inm363'
SRC = os.path.join(D, 'checkpoints', '006_COLAB_FULL_B05_sft_grpo_seed1')
DST = os.path.join(D, 'checkpoints', '018_ext_B05_seed1')
man_path = os.path.join(D, 'results', '018_ext_B05_seed1_step1000', 'continuation_manifest.json')

# Every state file the trainer needs to continue the exact run. A missing
# optimizer or scheduler would silently turn this into a different experiment
REQUIRED = ['adapter_model.safetensors', 'adapter_config.json', 'optimizer.pt',
            'scheduler.pt', 'trainer_state.json']

cks = [int(p.rsplit('-', 1)[1]) for p in glob.glob(os.path.join(DST, 'grpo', 'checkpoint-*'))]
if os.path.exists(man_path):
    # A resume is only valid if the manifest proves this directory belongs to the
    # recorded continuation. Checkpoint-500 itself may have rotated out, so what gets
    # verified is what persists: identity, the merged base, the config hash
    man = json.load(open(man_path))
    assert man['source_run'] == '006_COLAB_FULL_B05_sft_grpo_seed1', 'manifest names a different source run'
    for k, v in man['hashes'].items():
        if k.startswith('sft_merged/'):
            f = os.path.join(DST, k)
            assert os.path.exists(f) and sha(f) == v, f'{k}: merged base drifted since seeding'
    # the config hash must already exist and match, a missing file would get
    # silently recreated by the trainer wrapper instead of rejected
    import sys
    sys.path.insert(0, os.getcwd())
    from config import load_config
    from checkpoint import _config_hash
    hp = os.path.join(DST, 'grpo', 'config_hash.txt')
    assert os.path.exists(hp), 'resume without a config hash file, unrelated or damaged directory'
    cfg018 = load_config('configs/018_ext_colab_b05.yaml')
    assert open(hp).read().strip() == _config_hash(cfg018), 'config hash differs from the 018 config'
    assert cks, 'manifest exists but no checkpoint at all, damaged directory, stopping'
    # restarting before checkpoint-550 exists simply repeats steps 501-549
    print(f'resume verified against the manifest and config hash, newest checkpoint {max(cks)}')
else:
    # First execution: the destination must not contain surprises
    # without a manifest, require an empty destination. A partial seeding
    # (interrupted mid-copy) also lands here and gets rejected: delete the
    # 018 dir and rerun this cell
    existing = glob.glob(os.path.join(DST, 'grpo', '*'))
    assert not existing, f'files present but no continuation manifest, delete {DST} and rerun: {existing}'
    # stale evaluation artifacts would be skipped by the evaluator later and
    # compared as if they came from the new model, so a fresh run refuses them
    res = os.path.join(D, 'results', '018_ext_B05_seed1_step1000')
    stale = [f for f in ('grpo_eval.json', 'grpo_preds.jsonl', 'grpo_logprob.json', 'grpo_logprob_preds.jsonl')
             if os.path.exists(os.path.join(res, f))]
    assert not stale, f'evaluation artifacts already present on a fresh run: {stale}'

    ck_src = os.path.join(SRC, 'grpo', 'checkpoint-500')
    for fname in REQUIRED:
        assert os.path.exists(os.path.join(ck_src, fname)), f'source checkpoint missing {fname}, cannot continue'
    rng = [f for f in os.listdir(ck_src) if f.startswith('rng_state')]
    assert rng, 'source checkpoint has no rng state file, cannot continue the exact run'

    hashes = {}
    for sub in ('sft_merged',):
        src_dir, dst_dir = os.path.join(SRC, sub), os.path.join(DST, sub)
        os.makedirs(dst_dir, exist_ok=True)
        for f in sorted(os.listdir(src_dir)):
            s, d = os.path.join(src_dir, f), os.path.join(dst_dir, f)
            if not os.path.exists(d):
                shutil.copy2(s, d)
            hs, hd = sha(s), sha(d)
            assert hs == hd, f'{sub}/{f}: copy hash differs'
            hashes[f'{sub}/{f}'] = hs
    ck_dst = os.path.join(DST, 'grpo', 'checkpoint-500')
    os.makedirs(ck_dst, exist_ok=True)
    for f in sorted(os.listdir(ck_src)):
        s, d = os.path.join(ck_src, f), os.path.join(ck_dst, f)
        if not os.path.exists(d):
            shutil.copy2(s, d)
        hs, hd = sha(s), sha(d)
        assert hs == hd, f'checkpoint-500/{f}: copy hash differs'
        hashes[f'checkpoint-500/{f}'] = hs
    gs = json.load(open(os.path.join(ck_dst, 'trainer_state.json')))['global_step']
    assert gs == 500, f'seeded checkpoint says step {gs}'

    # the config hash is written as part of this same transaction, before the
    # manifest: the resume branch requires it, so it must never be possible for
    # the manifest to exist without the hash
    import sys
    sys.path.insert(0, os.getcwd())
    from config import load_config
    from checkpoint import write_config_hash, _config_hash
    cfg018 = load_config('configs/018_ext_colab_b05.yaml')
    write_config_hash(os.path.join(DST, 'grpo'), cfg018)
    written = open(os.path.join(DST, 'grpo', 'config_hash.txt')).read().strip()
    assert written == _config_hash(cfg018), 'config hash write-back verification failed'
    man = {'source_run': '006_COLAB_FULL_B05_sft_grpo_seed1', 'seeded': 'checkpoint-500 + sft_merged',
           'required_present': REQUIRED + rng, 'ts': time.strftime('%Y-%m-%d %H:%M'), 'hashes': hashes}
    os.makedirs(os.path.dirname(man_path), exist_ok=True)
    with open(man_path, 'w') as f:
        json.dump(man, f, indent=2)
    print(f'seeded and verified {len(hashes)} files, all required state present')

resume verified against the manifest and config hash, newest checkpoint 1000


## 7. The continuation run

Direct call so the resumed trainer streams here. Wandb picks up the key file
via the env vars below. Expected ~4.5-5h for steps 501-1000.

In [8]:
import json, os, sys, time
sys.path.insert(0, os.getcwd())
import torch
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
if os.path.exists('wandb_api_key.txt'):
    os.environ.setdefault('WANDB_API_KEY', open('wandb_api_key.txt').read().strip())
os.environ.setdefault('WANDB_PROJECT', 'inm363')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

from config import load_config
from scripts.env_switch import resolve_device, apply_smoke
from train.grpo import run_grpo
from train.model_utils import resolve_dtype
from obs import _git_info, _versions, gpu_snapshot, append_ledger

D0 = '/content/drive/MyDrive/inm363'
parent = json.load(open(os.path.join(D0, 'results', '006_COLAB_FULL_B05_sft_grpo_seed1', 'manifest.json')))
parent_torch = parent['versions']['torch']
assert torch.__version__ == parent_torch, \
    f'torch {torch.__version__} differs from the parent run ({parent_torch}), the continuation must match'

cfg = apply_smoke(load_config('configs/018_ext_colab_b05.yaml'))
device = resolve_device(cfg.env)
D = cfg.output_root
out_dir = os.path.join(D, 'checkpoints', '018_ext_B05_seed1', 'grpo')
res_dir = os.path.join(D, 'results', '018_ext_B05_seed1_step1000')

import glob as _glob

def newest_step():
    cks = _glob.glob(os.path.join(out_dir, 'checkpoint-*'))
    return max((int(c.rsplit('-', 1)[1]) for c in cks), default=0)

man_path = os.path.join(res_dir, 'continuation_manifest.json')
man = json.load(open(man_path))
start_step = newest_step()

# A hard runtime death cannot run finally, so any attempt left open by a
# previous session gets closed here first, whatever step things stopped at.
# The units session carries its billing, elapsed time is unavailable after a hard runtime stop
for a in man.get('attempts', []):
    if 'ts_end' not in a:
        a.update({'interrupted': True, 'end_step': start_step,
                  'elapsed_sec': None, 'ts_end': 'runtime died before finally'})
with open(man_path, 'w') as f:
    json.dump(man, f, indent=2)

# checkpoint-1000 alone is not finalisation: the root adapter, its provenance
# file and the trainlog are all written after it
final_artifacts = [os.path.join(out_dir, 'adapter_model.safetensors'),
                   os.path.join(out_dir, 'adapter_base.json'),
                   os.path.join(res_dir, 'grpo_trainlog.json')]
finalized = start_step >= 1000 and all(os.path.exists(f) for f in final_artifacts)

if finalized:
    print('step 1000 reached and all final artifacts present, training skipped, no new ledger entry')
else:
    if start_step >= 1000:
        print('checkpoint-1000 exists but finalisation is incomplete, running the finalisation pass')
    # record the environment before training starts, so checkpoints from an
    # interrupted attempt keep their provenance (a hard runtime death never
    # reaches finally)
    env = {'git': _git_info(), 'versions': _versions(),
           'resolved_dtype': resolve_dtype(cfg), 'ladder': 'disabled'}
    for a in man.get('attempts', []):
        prev = a.get('env')
        if prev:
            assert prev['git'].get('commit') == env['git'].get('commit'), \
                'git commit differs from the previous attempt, continue on the same commit'
            for lib in ('torch', 'trl', 'transformers', 'peft'):
                assert prev['versions'].get(lib) == env['versions'].get(lib), \
                    f'{lib} version differs from the previous attempt'
            assert prev['resolved_dtype'] == env['resolved_dtype'], 'dtype differs from the previous attempt'
    attempt = {'start_step': start_step, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
               'gpu': gpu_snapshot(), 'env': env, 'completed': False}
    man.setdefault('attempts', []).append(attempt)
    with open(man_path, 'w') as f:
        json.dump(man, f, indent=2)

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        # adapter_path points at the 006 sft adapter but the merge is skipped
        # because the seeded sft_merged already exists, so training runs on the
        # exact copy. Keep the fallback disabled so an oom stops
        # the run instead of mutating it. At step 1000 the trainer
        # returns immediately and only the final artifacts get written
        run_grpo(cfg, device,
                 adapter_path=os.path.join(D, 'checkpoints', '006_COLAB_FULL_B05_sft_grpo_seed1', 'sft'),
                 out_dir=out_dir, report_to=['wandb'], log_dir=res_dir, run_name='018_ext_seed1',
                 allow_oom_fallback=False)
    finally:
        wall = time.time() - t0
        end_step = newest_step()
        attempt.update({'end_step': end_step, 'ts_end': time.strftime('%Y-%m-%d %H:%M'),
                        'elapsed_sec': round(wall, 1),
                        'completed': end_step >= 1000 and all(os.path.exists(f) for f in final_artifacts),
                        'peak_vram_gib': round(torch.cuda.max_memory_reserved() / 1024**3, 2)})
        man['training'] = {'parent_git': parent['git'], 'parent_versions': parent['versions'],
                           'resolved_config': 'configs/018_ext_colab_b05.yaml',
                           'note': 'per-attempt environments live in attempts[].env'}
        with open(man_path, 'w') as f:
            json.dump(man, f, indent=2)
        append_ledger(os.path.join(D, 'results', 'compute_ledger.jsonl'),
                      {'run_tag': '018_ext_B05_seed1', 'condition': 'CONTINUATION',
                       'timings_sec': {'grpo_train_phase2': round(wall, 1)},
                       'train_steps': end_step - start_step,
                       'completed': attempt['completed'], 'gpu': gpu_snapshot()})
    print(f'attempt covered steps {start_step}-{end_step} in {wall/3600:.2f}h, manifest and ledger written')

step 1000 reached and all final artifacts present, training skipped, no new ledger entry


## 8. Completion gates

The combined reward history must cover steps 1-1000 exactly once each, the
restored 1-500 records must equal the originals, the resumed lr sequence must
match the measured scheduler boundary pattern, and the final state must say 1000.

In [9]:
import json, os
D = '/content/drive/MyDrive/inm363'
orig = json.load(open(os.path.join(D, 'results', '006_COLAB_FULL_B05_sft_grpo_seed1', 'grpo_trainlog.json')))
cont = json.load(open(os.path.join(D, 'results', '018_ext_B05_seed1_step1000', 'grpo_trainlog.json')))

def reward_list(log):
    # a list, not a dict, so duplicate steps cannot hide by overwriting
    return [(e['step'], e) for e in log if 'rewards/accuracy_reward/mean' in e and 'step' in e]

ol, cl = reward_list(orig), reward_list(cont)
from collections import Counter
oc, cc = Counter(s for s, _ in ol), Counter(s for s, _ in cl)
assert all(v == 1 for v in oc.values()), f'duplicate steps in the original log: {[s for s,v in oc.items() if v>1][:5]}'
assert all(v == 1 for v in cc.values()), f'duplicate steps in the continuation log: {[s for s,v in cc.items() if v>1][:5]}'
assert set(oc) == set(range(1, 501)), 'original log must cover steps 1..500 exactly'
restored = {s: e for s, e in cl if s <= 500}
assert set(restored) == set(range(1, 501)), 'restored history must cover steps 1..500 exactly'
o = dict(ol)
# restored steps are compared against the originals, never silently dropped
for s, e in restored.items():
    assert abs(e['rewards/accuracy_reward/mean'] - o[s]['rewards/accuracy_reward/mean']) < 1e-9, \
        f'restored step {s} reward differs from the original log'
phase2 = {s: e for s, e in cl if s > 500}
assert set(phase2) == set(range(501, 1001)), 'phase 2 must cover steps 501..1000 exactly'
combined = dict(o)
combined.update(phase2)

# the revived schedule, checked over every step: present, in range at the
# boundaries, and never increasing along the way
lrs = [(s, phase2[s]['learning_rate']) for s in range(501, 1001) if 'learning_rate' in phase2[s]]
assert len(lrs) == 500, f'expected 500 lr records, found {len(lrs)}'
# Boundary behaviour measured on the first attempt: step 501 applies the
# restored lr from the checkpoint (zero, the old schedule was exhausted) and
# the rebuilt 1000-step schedule takes over from step 502. So the phase has
# exactly one zero-lr step followed by 499 revived updates
assert lrs[0][1] == 0.0, f'step 501 expected the restored zero, saw {lrs[0][1]}'
assert 4e-6 <= lrs[1][1] <= 6e-6, f'step 502 lr {lrs[1][1]} outside the revived range'
assert lrs[-1][1] <= 5e-7, f'final lr {lrs[-1][1]} not near zero'
for (s1, a), (s2, b) in zip(lrs[1:], lrs[2:]):
    assert b <= a + 1e-12, f'lr increased between steps {s1} and {s2}'
nonzero = sum(1 for _, v in lrs if v > 0)
assert nonzero == 499, f'expected 499 nonzero-lr steps, found {nonzero}'
print(f'lr: one restored-zero step at 501, revived {lrs[1][1]:.2e} at 502 down to '
      f'{lrs[-1][1]:.2e} at 1000, non-increasing, 499 real updates')

import glob
final_ck = sorted(glob.glob(os.path.join(D, 'checkpoints', '018_ext_B05_seed1', 'grpo', 'checkpoint-*')),
                  key=lambda p: int(p.rsplit('-', 1)[1]))[-1]
gs = json.load(open(os.path.join(final_ck, 'trainer_state.json')))['global_step']
assert gs == 1000, f'final global_step {gs}'
with open(os.path.join(D, 'results', '018_ext_B05_seed1_step1000', 'combined_reward_curve.json'), 'w') as f:
    json.dump({str(s): combined[s]['rewards/accuracy_reward/mean'] for s in sorted(combined)}, f)
print('all completion gates passed, combined curve saved')

lr: one restored-zero step at 501, revived 4.99e-06 at 502 down to 1.00e-08 at 1000, non-increasing, 499 real updates
all completion gates passed, combined curve saved


## 9. Endpoint evaluation

In [10]:
import hashlib, json, os

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

D = '/content/drive/MyDrive/inm363'
# the evaluation is bound to the adapter hash: a skip is only alowed when the
# existing outputs provably came from this exact final adapter
adapter_sha = sha(os.path.join(D, 'checkpoints', '018_ext_B05_seed1', 'grpo', 'adapter_model.safetensors'))
out = os.path.join(D, 'results', '018_ext_B05_seed1_step1000')
bind_path = os.path.join(out, 'eval_binding.json')
if os.path.exists(os.path.join(out, 'grpo_eval.json')):
    assert os.path.exists(bind_path), 'existing eval outputs without a binding file, delete them and rerun'
    assert json.load(open(bind_path))['adapter_sha256'] == adapter_sha, \
        'existing eval outputs came from a different adapter, delete them and rerun'
else:
    os.makedirs(out, exist_ok=True)
    with open(bind_path, 'w') as f:
        json.dump({'adapter_sha256': adapter_sha}, f)
!python scripts/eval_adapter.py --config configs/018_ext_colab_b05.yaml --adapter {D}/checkpoints/018_ext_B05_seed1/grpo --condition grpo --out {D}/results/018_ext_B05_seed1_step1000 --seed 1
if _exit_code != 0:
    raise SystemExit(f'evaluation failed with exit code {_exit_code}')
import json, os
ev = json.load(open(os.path.join(D, 'results', '018_ext_B05_seed1_step1000', 'grpo_eval.json')))
lp = json.load(open(os.path.join(D, 'results', '018_ext_B05_seed1_step1000', 'grpo_logprob.json')))
assert ev['n_items'] == 1884 and lp['n_items'] == 1884, 'item counts wrong'
print('evaluation complete at 1884 items on both metrics')

README.md: 100% 53.2k/53.2k [00:00<00:00, 82.2MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 178MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00:00<?, ?B/s]
high_school_psychology/test-00000-of-000(…): downloading bytes: 100% 91.9k/91.9k [00:01<00:00, 64.9kB/s, 8.93kB/s  ]
high_school_psychology/test-00000-of-000(…): reconstructing file: 100% 92.8k/92.8k [00:01<00:00, 65.5kB/s, 9.01kB/s  ]

high_school_psychology/validation-00000-(…): downloading bytes:   0% 0.00/15.2k [00:00<?, ?B/s]
high_school_psychology/validation-00000-(…): downloading bytes: 100% 13.4k/13.4k [00:01<00:00, 10.2kB/s, 1.30kB/s  ]
high_school_psychology/validation-00000-(…): reconstructing file: 100% 15.2k/15.2k [00:01<00:00, 11.6kB/s, 1.48kB/s  ]

high_school_psychology/dev-00000-of-0000(…): downloading bytes:   0% 0.00/5.18k [00:00<?, ?B/s]
high_school_psychology/dev-00000-of-0000(…): downloading bytes: 100% 3.21k/3.21k [00:01<00:00, 2.49kB/s,   313B/s  ]
high_schoo

## 10. The pre-registered comparison

In [11]:
import json, os, sys
sys.path.insert(0, os.getcwd())
from analysis.stats import _read_preds, align_pred_rows, mcnemar_exact, paired_delta_ci
from data.subjects import tier_of, TIERS

D = '/content/drive/MyDrive/inm363'
run = '006_COLAB_FULL_B05_sft_grpo_seed1'
ndir = os.path.join(D, 'results', '018_ext_B05_seed1_step1000')
odir = os.path.join(D, 'results', run)
new = _read_preds(os.path.join(ndir, 'grpo_preds.jsonl'))
old = _read_preds(os.path.join(odir, 'grpo_preds.jsonl'))
ra, rb = align_pred_rows(new, old)
ca, cb = [r['correct'] for r in ra], [r['correct'] for r in rb]
delta = sum(ca) / len(ca) - sum(cb) / len(cb)
n01 = sum(1 for x, y in zip(ca, cb) if not x and y)
n10 = sum(1 for x, y in zip(ca, cb) if x and not y)
p = mcnemar_exact(n01, n10)
d, lo, hi = paired_delta_ci(cb, ca)   # step-1000 minus step-500

curve = json.load(open(os.path.join(ndir, 'combined_reward_curve.json')))
late = [curve[str(s)] for s in range(901, 1001)]
early = [curve[str(s)] for s in range(501, 601)]
rchange = sum(late) / len(late) - sum(early) / len(early)
rlabel = 'rising' if rchange >= 0.03 else ('declining' if rchange <= -0.03 else 'flat')

plp = _read_preds(os.path.join(ndir, 'grpo_logprob_preds.jsonl'))
flp = _read_preds(os.path.join(odir, 'grpo_logprob_preds.jsonl'))
pa_, pb_ = align_pred_rows(plp, flp)
pd_, plo, phi = paired_delta_ci([r['correct'] for r in pb_], [r['correct'] for r in pa_])
probe_tiers = {}
for t in TIERS:
    xs = [(x['correct'], y['correct']) for x, y in zip(pa_, pb_) if tier_of(x['subject']) == t]
    probe_tiers[t] = (sum(a for a, _ in xs) / len(xs), sum(b for _, b in xs) / len(xs))

nev = json.load(open(os.path.join(ndir, 'grpo_eval.json')))
oev = json.load(open(os.path.join(odir, 'grpo_eval.json')))

print(f'step-1000 {sum(ca)/len(ca):.3f} vs step-500 {sum(cb)/len(cb):.3f}  delta {delta:+.3f}  '
      f'p {p:.3g}  ci [{lo:+.3f}, {hi:+.3f}]')
tiers_out = {}
for t in TIERS:
    xs = [(x, y) for x, y, r in zip(ca, cb, ra) if tier_of(r['subject']) == t]
    tiers_out[t] = (sum(x for x, _ in xs) / len(xs), sum(y for _, y in xs) / len(xs))
    print(f'   {t}: {tiers_out[t][0]:.3f} vs {tiers_out[t][1]:.3f}')
print(f"fmt {nev['format_compliance']:.3f} vs {oev['format_compliance']:.3f}  "
      f"len {nev['mean_len']:.0f} vs {oev['mean_len']:.0f}  "
      f"xfail {nev['extraction_failure_rate']:.3f} vs {oev['extraction_failure_rate']:.3f}")
print(f'phase-2 reward change {rchange:+.3f} -> {rlabel}')
print(f'probe delta {pd_:+.3f}  ci [{plo:+.3f}, {phi:+.3f}]  per tier ' +
      '  '.join(f'{t} {a:.3f}/{b:.3f}' for t, (a, b) in probe_tiers.items()))

# verdict straight from the plan's outcome table, material checked first since
# the two rules cannot both fire
if abs(delta) >= 0.02 and p < 0.05:
    if delta > 0:
        verdict = ('material improvement, reward ' + rlabel + ': ' +
                   ('the restarted continuation helped this artifact' if rlabel == 'rising'
                    else "extends 017's reward-vs-test disconnect, gains without a reward signal"))
    else:
        verdict = 'material decline: the continuation harmed this artifact'
elif lo > -0.02 and hi < 0.02:
    verdict = 'equivalent: the recovery procedure produced no practically meaningful endpoint change'
else:
    verdict = 'inconclusive: no material change detected but not equivalent, prediction not contradicted'
print()
print('VERDICT (plan outcome table):', verdict)

# compact artifact so the memo and figures read numbers, not console history
summary = {'delta': delta, 'p': p, 'ci': [lo, hi], 'tiers': tiers_out,
           'fmt': [nev['format_compliance'], oev['format_compliance']],
           'len': [nev['mean_len'], oev['mean_len']],
           'xfail': [nev['extraction_failure_rate'], oev['extraction_failure_rate']],
           'reward_change': rchange, 'reward_label': rlabel,
           'probe_delta': pd_, 'probe_ci': [plo, phi], 'probe_tiers': probe_tiers,
           'verdict': verdict}
with open(os.path.join(ndir, 'comparison_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('comparison summary saved')

step-1000 0.515 vs step-500 0.315  delta +0.200  p 1.2e-53  ci [+0.175, +0.224]
   easy: 0.658 vs 0.379
   medium: 0.476 vs 0.284
   hard: 0.258 vs 0.222
fmt 0.905 vs 0.898  len 79 vs 107  xfail 0.051 vs 0.059
phase-2 reward change +0.095 -> rising
probe delta -0.001  ci [-0.008, +0.006]  per tier easy 0.681/0.680  medium 0.516/0.512  hard 0.366/0.379

VERDICT (plan outcome table): material improvement, reward rising: the restarted continuation helped this artifact
comparison summary saved


## 11. Close the units record

In [13]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 17.65    # fill after the evaluation
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

logged: {'session_id': '20260727_162656', 'ts_start': '2026-07-27 16:27', 'notebook': '018_ext_seed1', 'colab_cu_before': 23.04, 'unit_rate': 5.3, 'ts_end': '2026-07-27 17:34', 'colab_cu_after': 17.65, 'colab_cu_used': 5.39, 'closed_on_resume': False}
